# SCG HVDNet Colab Debug Notebook

Step-by-step notebook for Task I/II/III with training, inference, and debug-first visualizations.

## Steps
1. Environment setup
2. Path and task configuration
3. Data loading and preprocessing utilities
4. Model definition (HVDNet)
5. Build dataset tensors
6. Train (optional) or load checkpoint
7. Inference and attention visualization
8. Confusion matrix and metrics

In [ ]:
# Step 1: (Optional) install dependencies in Colab
# Set INSTALL_DEPS=True only for a fresh runtime.
INSTALL_DEPS = False

if INSTALL_DEPS:
    import subprocess
    import sys

    subprocess.check_call([
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "numpy",
        "pandas",
        "scipy",
        "scikit-learn",
        "matplotlib",
        "torch",
    ])
    print("Dependencies installed.")

In [ ]:
# Step 2: imports, seed, and runtime device
import os
import json
import random
from dataclasses import dataclass

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset

from scipy import signal
from sklearn.model_selection import train_test_split, StratifiedKFold, KFold
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"DEVICE = {DEVICE}")

In [ ]:
# Step 3: Google Drive mount and project path config
import importlib

USE_DRIVE = True
DRIVE_ROOT = "/content/drive/MyDrive/SCG_MachineLearning"

if USE_DRIVE:
    colab_spec = importlib.util.find_spec("google.colab")
    if colab_spec is None:
        raise RuntimeError("google.colab is not available. Set USE_DRIVE=False outside Colab.")
    drive = importlib.import_module("google.colab.drive")
    drive.mount('/content/drive', force_remount=False)

PROJECT_ROOT = DRIVE_ROOT if USE_DRIVE else os.getcwd()
DATA_DIR = os.path.join(PROJECT_ROOT, "Data")
SAVED_PEAKS_DIR = os.path.join(PROJECT_ROOT, "Saved_Peaks")

CHECKPOINTS = {
    "Task I": os.path.join(PROJECT_ROOT, "hvdnet_task_i_1.pt"),
    "Task II": os.path.join(PROJECT_ROOT, "hvdnet_task_ii_1.pt"),
    "Task III": os.path.join(PROJECT_ROOT, "hvdnet_task_iii_1.pt"),
}

TASK_NAME = "Task I"  # choose: Task I, Task II, Task III
MAX_PATIENTS = 40         # reduce for fast debugging
MAX_SEGMENTS_PER_PATIENT = 30

print("PROJECT_ROOT:", PROJECT_ROOT)
print("DATA_DIR exists:", os.path.isdir(DATA_DIR))
print("SAVED_PEAKS_DIR exists:", os.path.isdir(SAVED_PEAKS_DIR))
for k, v in CHECKPOINTS.items():
    print(f"{k} checkpoint exists: {os.path.isfile(v)}")

In [ ]:
# Step 4: task mappings and preprocessing utilities
LABEL_COLUMNS = [
    "Moderate or greater MS",
    "Moderate or greater MR",
    "Moderate or greater AR",
    "Moderate or greater AS",
    "Moderate or greater TR",
]

TASK_CLASS_NAMES = {
    "Task I": ["AS", "MR", "MS", "AR", "N"],
    "Task II": ["AS", "AS-MR", "AS-MS", "AS-AR", "AS-TR"],
    "Task III": ["MS", "MR", "AR", "AS", "TR"],
}

@dataclass
class PatientRecord:
    patient_id: str
    fs: int
    signals: dict
    filtered_signals: dict
    r_peaks_indices: np.ndarray
    segments: list
    label_row: dict


def get_original_fs(patient_id: str) -> int:
    if patient_id.startswith('UP-'):
        try:
            num = int(patient_id.split('-')[1])
            if 22 <= num <= 30:
                return 512
        except ValueError:
            pass
    return 256


def time_to_seconds(time_str: str) -> float:
    h, m, s = time_str.split(':')
    return int(h) * 3600 + int(m) * 60 + float(s)


def load_peaks_json(patient_id: str, signal_length: int, annotation_source: str = "AO"):
    annotation_source = (annotation_source or "AO").upper()
    if annotation_source == "AO":
        json_path = os.path.join(SAVED_PEAKS_DIR, f"{patient_id}_AO_Peaks.json")
        key_name = f"{patient_id}_AO_Peaks"
    else:
        json_path = os.path.join(DATA_DIR, f"{patient_id}-ECG.json")
        key_name = "LARA_R_Peaks"

    if not os.path.isfile(json_path):
        return None

    with open(json_path, "r") as f:
        peak_data = json.load(f)

    time_strings = peak_data.get(key_name)
    if time_strings is None:
        time_strings = next(iter(peak_data.values()), [])

    peak_seconds = [time_to_seconds(ts) for ts in time_strings]
    peak_indices = [int(np.round(sec * 256)) for sec in peak_seconds]
    peak_indices = [idx for idx in peak_indices if 0 <= idx < signal_length]
    return np.asarray(peak_indices, dtype=int)


def detect_ecg_peaks_fallback(ecg_signal: np.ndarray, fs: int = 256):
    distance = int(0.27 * fs)
    prominence = max(np.std(ecg_signal) * 0.5, 1e-3)
    peaks, _ = signal.find_peaks(ecg_signal, distance=distance, prominence=prominence)
    return peaks.astype(int)


def apply_zero_phase_butterworth(signals_dict, fs, lowcut=1.0, highcut=30.0, order=6):
    nyquist = fs / 2.0
    if not (0 < lowcut < highcut < nyquist):
        raise ValueError(f"Invalid bandpass range: {lowcut}-{highcut} Hz for fs={fs}")

    b, a = signal.butter(order, [lowcut, highcut], btype='bandpass', fs=fs)
    filtered = {}
    for name, values in signals_dict.items():
        filtered[name] = signal.filtfilt(b, a, np.asarray(values, dtype=float))
    return filtered


def build_rpeak_segments(r_peaks, signal_length):
    segments = []
    for i in range(len(r_peaks) - 3):
        start_idx = int(r_peaks[i])
        end_idx = int(r_peaks[i + 3])
        if 0 <= start_idx < end_idx <= signal_length:
            segments.append({
                'segment_id': i,
                'start_idx': start_idx,
                'end_idx': end_idx,
                'start_peak_number': i,
                'end_peak_number': i + 3,
            })
    return segments


def zscore_normalize(values):
    values = np.asarray(values, dtype=float)
    mean_val = np.mean(values)
    std_val = np.std(values)
    if std_val < 1e-12:
        return np.zeros_like(values)
    return (values - mean_val) / std_val


def pad_or_truncate(values, target_len=800):
    values = np.asarray(values, dtype=float)
    if len(values) < target_len:
        return np.pad(values, (0, target_len - len(values)), mode='constant')
    if len(values) > target_len:
        return values[:target_len]
    return values


def label_row_to_multihot(label_row):
    return np.array([
        float(int(label_row.get(label_name, 0)))
        for label_name in LABEL_COLUMNS
    ], dtype=np.float32)


def map_label_row_to_task_index(label_row, task_name):
    ms = int(label_row.get("Moderate or greater MS", 0))
    mr = int(label_row.get("Moderate or greater MR", 0))
    ar = int(label_row.get("Moderate or greater AR", 0))
    as_val = int(label_row.get("Moderate or greater AS", 0))
    tr = int(label_row.get("Moderate or greater TR", 0))

    total_positive = ms + mr + ar + as_val + tr

    if task_name == "Task I":
        if total_positive == 0:
            return 4
        if tr == 1:
            return None
        if (as_val + mr + ms + ar) != 1:
            return None
        if as_val == 1:
            return 0
        if mr == 1:
            return 1
        if ms == 1:
            return 2
        if ar == 1:
            return 3
        return None

    if task_name == "Task II":
        if as_val != 1:
            return None
        coexisting_count = mr + ms + ar + tr
        if coexisting_count == 0:
            return 0
        if coexisting_count == 1:
            if mr == 1:
                return 1
            if ms == 1:
                return 2
            if ar == 1:
                return 3
            if tr == 1:
                return 4
        return None

    if task_name == "Task III":
        return label_row_to_multihot(label_row)

    raise ValueError(f"Unknown task: {task_name}")


def load_patient_record(patient_id: str, annotation_source: str = "AO") -> PatientRecord:
    csv_path = os.path.join(DATA_DIR, f"Cleaned_{patient_id}.csv")
    if not os.path.isfile(csv_path):
        raise FileNotFoundError(f"Missing file: {csv_path}")

    df = pd.read_csv(csv_path)
    original_fs = get_original_fs(patient_id)

    accx = df['AccX'].values
    accy = df['AccY'].values
    accz = df['AccZ'].values
    ecg = df['ECG'].values

    if original_fs == 512:
        new_len = len(accx) // 2
        accx = signal.resample(accx, new_len)
        accy = signal.resample(accy, new_len)
        accz = signal.resample(accz, new_len)
        ecg = signal.resample(ecg, new_len)

    signals_dict = {
        'AccX': accx,
        'AccY': accy,
        'AccZ': accz,
        'ECG': ecg,
    }

    filtered = apply_zero_phase_butterworth(signals_dict, 256, lowcut=1.0, highcut=30.0, order=6)

    peaks = load_peaks_json(patient_id, signal_length=len(accx), annotation_source=annotation_source)
    if peaks is None or len(peaks) < 4:
        peaks = detect_ecg_peaks_fallback(filtered['ECG'], fs=256)

    segments = build_rpeak_segments(peaks, len(accx))

    labels_path = os.path.join(DATA_DIR, "ground_truth_labels.csv")
    labels_df = pd.read_csv(labels_path)
    labels_df.columns = [c.strip().replace('﻿', '') for c in labels_df.columns]
    row = labels_df[labels_df['Patient ID'] == patient_id]
    if row.empty:
        raise ValueError(f"No labels found for patient {patient_id}")
    label_row = row.iloc[0].to_dict()

    return PatientRecord(
        patient_id=patient_id,
        fs=256,
        signals=signals_dict,
        filtered_signals=filtered,
        r_peaks_indices=peaks,
        segments=segments,
        label_row=label_row,
    )

In [ ]:
# Step 5: visualization helpers (raw/filtered/segment + attention + training curves)
def plot_signal_stages(record: PatientRecord, segment_idx: int = 0):
    fig, axes = plt.subplots(3, 1, figsize=(14, 8), sharex=False)
    channels = ['AccX', 'AccY', 'AccZ']

    for i, ch in enumerate(channels):
        axes[i].plot(record.signals[ch], alpha=0.45, label=f"{ch} raw")
        axes[i].plot(record.filtered_signals[ch], alpha=0.95, label=f"{ch} filtered")
        axes[i].set_title(f"{record.patient_id} - {ch}")
        axes[i].legend(loc='upper right')

    if record.segments:
        seg = record.segments[min(segment_idx, len(record.segments) - 1)]
        for ax in axes:
            ax.axvspan(seg['start_idx'], seg['end_idx'], color='orange', alpha=0.15, label='segment')

    plt.tight_layout()
    plt.show()


def plot_training_curves(history):
    if not history['epoch']:
        print('No training history available.')
        return
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].plot(history['epoch'], history['train_loss'], label='train loss')
    axes[0].plot(history['epoch'], history['val_loss'], label='val loss')
    axes[0].set_title('Loss curve')
    axes[0].legend()

    axes[1].plot(history['epoch'], history['train_acc'], label='train acc')
    axes[1].plot(history['epoch'], history['val_acc'], label='val acc')
    axes[1].set_title('Accuracy curve')
    axes[1].legend()

    plt.tight_layout()
    plt.show()


def plot_attention_overlay(signal_1d, attention_1d, title):
    signal_1d = np.asarray(signal_1d).reshape(-1)
    attention_1d = np.asarray(attention_1d).reshape(-1)
    x = np.arange(len(signal_1d))

    plt.figure(figsize=(12, 4))
    plt.plot(x, signal_1d, color='black', linewidth=1.2, label='signal')
    attn_norm = (attention_1d - attention_1d.min()) / (attention_1d.ptp() + 1e-8)
    plt.fill_between(x, signal_1d.min(), signal_1d.max(), where=attn_norm > 0.5, alpha=0.25, color='red', label='high attention')
    plt.title(title)
    plt.legend()
    plt.tight_layout()
    plt.show()

In [ ]:
# Step 6: model definition (taken from machinelearning.py)
class sCNN_Block(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size):
        super().__init__()
        self.conv1 = nn.Conv1d(in_channels, out_channels, kernel_size, padding='same')
        self.bn1 = nn.BatchNorm1d(out_channels)
        self.relu = nn.ReLU()

        self.conv2 = nn.Conv1d(out_channels, out_channels, kernel_size, padding='same')
        self.bn2 = nn.BatchNorm1d(out_channels)

        self.conv3 = nn.Conv1d(out_channels, out_channels, kernel_size, padding='same')
        self.bn3 = nn.BatchNorm1d(out_channels)

        self.pool = nn.MaxPool1d(kernel_size=2)
        self.skip_projection = nn.Identity()
        if in_channels != out_channels:
            self.skip_projection = nn.Conv1d(in_channels, out_channels, kernel_size=1)

    def forward(self, x):
        identity = self.skip_projection(x)
        out = self.relu(self.bn1(self.conv1(x)))
        out = self.relu(self.bn2(self.conv2(out)))
        out = self.bn3(self.conv3(out))
        out = self.relu(out + identity)
        out = self.pool(out)
        return out


class sCNN_Module(nn.Module):
    def __init__(self, in_channels=1, base_filters=64, kernel_size=7):
        super().__init__()
        channels = (base_filters, base_filters // 2, base_filters // 4)
        blocks = []
        c_in = in_channels
        for c_out in channels:
            blocks.append(sCNN_Block(c_in, c_out, kernel_size))
            c_in = c_out
        self.blocks = nn.Sequential(*blocks)

    def forward(self, x):
        return self.blocks(x)


class LSTM_Module(nn.Module):
    def __init__(self, input_features=16, hidden_size=64):
        super().__init__()
        self.lstm = nn.LSTM(input_size=input_features, hidden_size=hidden_size, num_layers=1, batch_first=True)

    def forward(self, x):
        x = x.permute(0, 2, 1)
        out, _ = self.lstm(x)
        return out


class SA_Module(nn.Module):
    def __init__(self, hidden_size=64):
        super().__init__()
        self.dense = nn.Linear(hidden_size, 1)

    def forward(self, lstm_out):
        scores = torch.tanh(self.dense(lstm_out))
        weights = F.softmax(scores, dim=1)
        weighted_out = lstm_out * weights
        context_vector = torch.sum(weighted_out, dim=1)
        return context_vector, weights


class HVDNet(nn.Module):
    def __init__(self, num_classes=5, d=64):
        super().__init__()
        self.scnn_x = sCNN_Module(in_channels=1, base_filters=d)
        self.scnn_y = sCNN_Module(in_channels=1, base_filters=d)
        self.scnn_z = sCNN_Module(in_channels=1, base_filters=d)

        self.lstm_x = LSTM_Module(input_features=d // 4, hidden_size=d)
        self.lstm_y = LSTM_Module(input_features=d // 4, hidden_size=d)
        self.lstm_z = LSTM_Module(input_features=d // 4, hidden_size=d)

        self.sa_x = SA_Module(hidden_size=d)
        self.sa_y = SA_Module(hidden_size=d)
        self.sa_z = SA_Module(hidden_size=d)

        self.classifier = nn.Sequential(
            nn.Linear(3 * d, d),
            nn.ReLU(),
            nn.BatchNorm1d(d),
            nn.Dropout(0.2),
            nn.Linear(d, num_classes),
        )

    def forward(self, x, y, z):
        feat_x = self.scnn_x(x)
        feat_y = self.scnn_y(y)
        feat_z = self.scnn_z(z)

        lstm_x = self.lstm_x(feat_x)
        lstm_y = self.lstm_y(feat_y)
        lstm_z = self.lstm_z(feat_z)

        ctx_x, attn_x = self.sa_x(lstm_x)
        ctx_y, attn_y = self.sa_y(lstm_y)
        ctx_z, attn_z = self.sa_z(lstm_z)

        logits = self.classifier(torch.cat((ctx_x, ctx_y, ctx_z), dim=1))
        return logits, (attn_x, attn_y, attn_z)


NUM_CLASSES = len(TASK_CLASS_NAMES[TASK_NAME])
model = HVDNet(num_classes=NUM_CLASSES, d=64).to(DEVICE)
print(model.__class__.__name__, 'initialized with classes:', TASK_CLASS_NAMES[TASK_NAME])

In [ ]:
# Step 7: inspect one patient quickly (debug checkpoint)
patient_files = sorted([f for f in os.listdir(DATA_DIR) if f.startswith('Cleaned_') and f.endswith('.csv')])
patient_ids = [f.replace('Cleaned_', '').replace('.csv', '') for f in patient_files]
print('Total patients found:', len(patient_ids))
print('First 5 patient IDs:', patient_ids[:5])

debug_patient_id = patient_ids[0]
debug_record = load_patient_record(debug_patient_id, annotation_source='AO')
print(f'Patient: {debug_record.patient_id}')
print(
    f'Signal lengths: X={len(debug_record.signals["AccX"])}, '
    f'Y={len(debug_record.signals["AccY"])}, '
    f'Z={len(debug_record.signals["AccZ"])}, '
    f'ECG={len(debug_record.signals["ECG"])}'
)
print(f'Peaks count: {len(debug_record.r_peaks_indices)}')
print(f'Segments count: {len(debug_record.segments)}')
print('Label row preview:', {k: debug_record.label_row.get(k, None) for k in ['Patient ID'] + LABEL_COLUMNS})

plot_signal_stages(debug_record, segment_idx=0)

In [ ]:
# Step 8: dataset builder for selected task
def build_task_dataset(task_name: str, max_patients: int = None, max_segments_per_patient: int = 30):
    x_segments, y_segments, z_segments = [], [], []
    labels = []
    sample_meta = []

    selected_ids = patient_ids[:max_patients] if max_patients is not None else patient_ids

    skipped_patients = 0
    for pid in selected_ids:
        try:
            rec = load_patient_record(pid, annotation_source='AO')
        except Exception as e:
            skipped_patients += 1
            print(f'[SKIP] {pid}: {e}')
            continue

        if task_name == 'Task III':
            target = map_label_row_to_task_index(rec.label_row, task_name)
            if target is None:
                continue
        else:
            target = map_label_row_to_task_index(rec.label_row, task_name)
            if target is None:
                continue

        if len(rec.segments) == 0:
            continue

        use_segments = rec.segments[:max_segments_per_patient]
        for seg in use_segments:
            s, e = seg['start_idx'], seg['end_idx']
            x = pad_or_truncate(zscore_normalize(rec.filtered_signals['AccX'][s:e]), target_len=800)
            y = pad_or_truncate(zscore_normalize(rec.filtered_signals['AccY'][s:e]), target_len=800)
            z = pad_or_truncate(zscore_normalize(rec.filtered_signals['AccZ'][s:e]), target_len=800)

            x_segments.append(x)
            y_segments.append(y)
            z_segments.append(z)
            labels.append(target)
            sample_meta.append({'patient_id': pid, 'segment_id': seg['segment_id']})

    if len(labels) == 0:
        raise RuntimeError('No valid samples produced. Check paths/task filters.')

    x_np = np.asarray(x_segments, dtype=np.float32)[:, None, :]
    y_np = np.asarray(y_segments, dtype=np.float32)[:, None, :]
    z_np = np.asarray(z_segments, dtype=np.float32)[:, None, :]

    if task_name == 'Task III':
        y_label = np.asarray(labels, dtype=np.float32)
    else:
        y_label = np.asarray(labels, dtype=np.int64)

    print('Dataset summary:')
    print('  x shape:', x_np.shape)
    print('  y shape:', y_np.shape)
    print('  z shape:', z_np.shape)
    print('  labels shape:', y_label.shape)
    print('  skipped patients:', skipped_patients)
    if task_name != 'Task III':
        unique, counts = np.unique(y_label, return_counts=True)
        print('  class counts:', dict(zip(unique.tolist(), counts.tolist())))

    return x_np, y_np, z_np, y_label, sample_meta


x_np, y_np, z_np, labels_np, meta = build_task_dataset(
    TASK_NAME,
    max_patients=MAX_PATIENTS,
    max_segments_per_patient=MAX_SEGMENTS_PER_PATIENT,
)

In [ ]:
# Step 9: train/val/test split and dataloaders
BATCH_SIZE = 64
NUM_EPOCHS = 8   # increase for full training
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 4e-3
TEST_SIZE = 0.2
VAL_SIZE_FROM_TRAIN = 0.2

indices = np.arange(len(labels_np))
is_multilabel = TASK_NAME == 'Task III'

if is_multilabel:
    train_idx, test_idx = train_test_split(indices, test_size=TEST_SIZE, random_state=SEED, shuffle=True)
else:
    train_idx, test_idx = train_test_split(indices, test_size=TEST_SIZE, random_state=SEED, shuffle=True, stratify=labels_np)

if is_multilabel:
    tr_idx, val_idx = train_test_split(train_idx, test_size=VAL_SIZE_FROM_TRAIN, random_state=SEED, shuffle=True)
else:
    tr_idx, val_idx = train_test_split(train_idx, test_size=VAL_SIZE_FROM_TRAIN, random_state=SEED, shuffle=True, stratify=labels_np[train_idx])

def make_loader(idxs, shuffle=False):
    x_t = torch.tensor(x_np[idxs], dtype=torch.float32)
    y_t = torch.tensor(y_np[idxs], dtype=torch.float32)
    z_t = torch.tensor(z_np[idxs], dtype=torch.float32)
    if is_multilabel:
        l_t = torch.tensor(labels_np[idxs], dtype=torch.float32)
    else:
        l_t = torch.tensor(labels_np[idxs], dtype=torch.long)
    ds = TensorDataset(x_t, y_t, z_t, l_t)
    return DataLoader(ds, batch_size=BATCH_SIZE, shuffle=shuffle)

train_loader = make_loader(tr_idx, shuffle=True)
val_loader = make_loader(val_idx, shuffle=False)
test_loader = make_loader(test_idx, shuffle=False)

print(f'train={len(tr_idx)}, val={len(val_idx)}, test={len(test_idx)}')
print('single batch shapes:', next(iter(train_loader))[0].shape, next(iter(train_loader))[3].shape)

In [ ]:
# Step 10: training loop (optional if using checkpoint)
def evaluate(model, loader, criterion, multilabel=False):
    model.eval()
    total_loss, total_correct, total_count = 0.0, 0.0, 0
    all_preds, all_true = [], []

    with torch.no_grad():
        for bx, by, bz, bl in loader:
            bx, by, bz, bl = bx.to(DEVICE), by.to(DEVICE), bz.to(DEVICE), bl.to(DEVICE)
            logits, _ = model(bx, by, bz)
            loss = criterion(logits, bl)
            total_loss += loss.item() * bl.size(0)

            if multilabel:
                pred = (torch.sigmoid(logits) > 0.5).float()
                total_correct += (pred == bl).float().sum().item()
                total_count += bl.numel()
                all_preds.append(pred.cpu().numpy())
                all_true.append(bl.cpu().numpy())
            else:
                pred = torch.argmax(logits, dim=1)
                total_correct += (pred == bl).sum().item()
                total_count += bl.size(0)
                all_preds.append(pred.cpu().numpy())
                all_true.append(bl.cpu().numpy())

    return (
        total_loss / max(len(loader.dataset), 1),
        100.0 * total_correct / max(total_count, 1),
        np.concatenate(all_true),
        np.concatenate(all_preds),
    )


def train_model(model, train_loader, val_loader, epochs=8, lr=1e-3, wd=4e-3, multilabel=False):
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=wd)
    criterion = nn.BCEWithLogitsLoss() if multilabel else nn.CrossEntropyLoss()

    history = {
        'epoch': [],
        'train_loss': [],
        'train_acc': [],
        'val_loss': [],
        'val_acc': [],
    }

    best_val = float('inf')
    best_state = None

    for ep in range(1, epochs + 1):
        model.train()
        running_loss, running_correct, running_total = 0.0, 0.0, 0

        for bx, by, bz, bl in train_loader:
            bx, by, bz, bl = bx.to(DEVICE), by.to(DEVICE), bz.to(DEVICE), bl.to(DEVICE)
            optimizer.zero_grad()
            logits, _ = model(bx, by, bz)
            loss = criterion(logits, bl)
            loss.backward()
            optimizer.step()

            running_loss += loss.item() * bl.size(0)
            if multilabel:
                pred = (torch.sigmoid(logits) > 0.5).float()
                running_correct += (pred == bl).float().sum().item()
                running_total += bl.numel()
            else:
                pred = torch.argmax(logits, dim=1)
                running_correct += (pred == bl).sum().item()
                running_total += bl.size(0)

        train_loss = running_loss / max(len(train_loader.dataset), 1)
        train_acc = 100.0 * running_correct / max(running_total, 1)

        val_loss, val_acc, _, _ = evaluate(model, val_loader, criterion, multilabel=multilabel)

        history['epoch'].append(ep)
        history['train_loss'].append(train_loss)
        history['train_acc'].append(train_acc)
        history['val_loss'].append(val_loss)
        history['val_acc'].append(val_acc)

        if val_loss < best_val:
            best_val = val_loss
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

        print(f'Epoch {ep:02d}/{epochs} | train_loss={train_loss:.4f} train_acc={train_acc:.2f}% | val_loss={val_loss:.4f} val_acc={val_acc:.2f}%')

    if best_state is not None:
        model.load_state_dict(best_state)

    return model, history


TRAIN_FROM_SCRATCH = False
history = {'epoch': [], 'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}

if TRAIN_FROM_SCRATCH:
    model, history = train_model(
        model,
        train_loader,
        val_loader,
        epochs=NUM_EPOCHS,
        lr=LEARNING_RATE,
        wd=WEIGHT_DECAY,
        multilabel=is_multilabel,
    )
    plot_training_curves(history)
else:
    print('Skipping training (TRAIN_FROM_SCRATCH=False).')

In [ ]:
# Step 11: checkpoint loading and quick save utility
def load_checkpoint_into_model(model, ckpt_path):
    if not os.path.isfile(ckpt_path):
        raise FileNotFoundError(f'Missing checkpoint: {ckpt_path}')

    obj = torch.load(ckpt_path, map_location=DEVICE)
    if isinstance(obj, dict) and 'state_dict' in obj:
        state = obj['state_dict']
    else:
        state = obj

    missing, unexpected = model.load_state_dict(state, strict=False)
    print('Loaded checkpoint:', ckpt_path)
    print('  Missing keys:', len(missing))
    print('  Unexpected keys:', len(unexpected))


if not TRAIN_FROM_SCRATCH:
    try:
        load_checkpoint_into_model(model, CHECKPOINTS[TASK_NAME])
    except Exception as e:
        print('[WARN] Could not load checkpoint:', e)


SAVE_TRAINED_CHECKPOINT = False
if SAVE_TRAINED_CHECKPOINT and TRAIN_FROM_SCRATCH:
    save_path = os.path.join(PROJECT_ROOT, f'hvdnet_{TASK_NAME.lower().replace(" ", "_")}_colab_debug.pt')
    torch.save(model.state_dict(), save_path)
    print('Saved:', save_path)

In [ ]:
# Step 12: inference + attention overlay
model.eval()

sample_batch = next(iter(test_loader))
bx, by, bz, bl = [t.to(DEVICE) for t in sample_batch]

with torch.no_grad():
    logits, (attn_x, attn_y, attn_z) = model(bx, by, bz)

if is_multilabel:
    probs = torch.sigmoid(logits)
    pred = (probs > 0.5).float()
    print('Task III multi-label prediction (first sample):')
    print('  probs:', probs[0].detach().cpu().numpy())
    print('  pred :', pred[0].detach().cpu().numpy())
    print('  true :', bl[0].detach().cpu().numpy())
else:
    probs = torch.softmax(logits, dim=1)
    pred_idx = int(torch.argmax(probs[0]).item())
    true_idx = int(bl[0].item())
    print('Prediction (first sample):')
    print('  pred idx/name:', pred_idx, TASK_CLASS_NAMES[TASK_NAME][pred_idx])
    print('  true idx/name:', true_idx, TASK_CLASS_NAMES[TASK_NAME][true_idx])
    print('  probs:', probs[0].detach().cpu().numpy())

sig_x = bx[0, 0].detach().cpu().numpy()
sig_y = by[0, 0].detach().cpu().numpy()
sig_z = bz[0, 0].detach().cpu().numpy()

attn_x_np = attn_x[0].detach().cpu().numpy().reshape(-1)
attn_y_np = attn_y[0].detach().cpu().numpy().reshape(-1)
attn_z_np = attn_z[0].detach().cpu().numpy().reshape(-1)

# Attention is over reduced temporal dimension; upsample to signal length for overlay.
attn_x_up = signal.resample(attn_x_np, len(sig_x))
attn_y_up = signal.resample(attn_y_np, len(sig_y))
attn_z_up = signal.resample(attn_z_np, len(sig_z))

plot_attention_overlay(sig_x, attn_x_up, f'{TASK_NAME} - Attention Overlay AccX')
plot_attention_overlay(sig_y, attn_y_up, f'{TASK_NAME} - Attention Overlay AccY')
plot_attention_overlay(sig_z, attn_z_up, f'{TASK_NAME} - Attention Overlay AccZ')

In [ ]:
# Step 13: evaluation metrics + confusion matrix
criterion = nn.BCEWithLogitsLoss() if is_multilabel else nn.CrossEntropyLoss()
test_loss, test_acc, y_true, y_pred = evaluate(model, test_loader, criterion, multilabel=is_multilabel)
print(f'Test loss: {test_loss:.4f}')
print(f'Test acc : {test_acc:.2f}%')

if not is_multilabel:
    class_names = TASK_CLASS_NAMES[TASK_NAME]
    cm = confusion_matrix(y_true, y_pred, labels=list(range(len(class_names))))

    plt.figure(figsize=(6, 5))
    plt.imshow(cm, interpolation='nearest', cmap='Blues')
    plt.title(f'Confusion Matrix - {TASK_NAME}')
    plt.colorbar()
    ticks = np.arange(len(class_names))
    plt.xticks(ticks, class_names, rotation=45)
    plt.yticks(ticks, class_names)

    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            plt.text(j, i, str(cm[i, j]), ha='center', va='center', color='black')

    plt.ylabel('True label')
    plt.xlabel('Predicted label')
    plt.tight_layout()
    plt.show()

    print(classification_report(y_true, y_pred, target_names=class_names, digits=4))
else:
    # Multi-label summary
    y_true_bin = y_true.astype(int)
    y_pred_bin = y_pred.astype(int)

    per_label_acc = (y_true_bin == y_pred_bin).mean(axis=0)
    for i, name in enumerate(TASK_CLASS_NAMES[TASK_NAME]):
        print(f'{name}: elementwise-acc={per_label_acc[i]:.4f}')
    print('Exact-match acc:', np.mean(np.all(y_true_bin == y_pred_bin, axis=1)))

## Troubleshooting
- If Data paths fail: verify PROJECT_ROOT and whether Drive is mounted.
- If no segments: check peaks JSON availability or rely on ECG fallback peaks.
- If checkpoint load mismatch: keep strict=False (already set) and ensure task matches checkpoint.
- If CUDA OOM: reduce BATCH_SIZE and MAX_SEGMENTS_PER_PATIENT.
- If class imbalance causes unstable training: lower LR and increase patient count.